<a href="https://colab.research.google.com/github/Khalidsyfullah/USplitVQA/blob/main/Direct_Model_Apply/New_BioMedClip_Centralized_SLAKE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "open_clip_torch", "transformers", "datasets", "openpyxl", "tqdm"])

import os, json, random, time
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = "/content/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# CONFIG
# =============================================================================
class Config:
    clip_model_name = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    vision_dim = 768; text_dim = 768; hidden_dim = 768
    num_fusion_layers = 2; num_attn_heads = 8; fusion_dropout = 0.25
    epochs = 30; batch_size = 16; learning_rate = 2e-5
    weight_decay = 1e-4; label_smoothing = 0.1
    early_stopping_patience = 8; lr_reduce_patience = 4
    lr_reduce_factor = 0.5; min_lr = 1e-7
    freeze_clip_epochs = 4; encoder_lr_scale = 0.1
    val_split = 0.15; max_answer_vocab = 0; min_answer_freq = 2

cfg = Config()


# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Modality synonyms (SLAKE has CT/MRI/X-Ray) ──
    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    # ── Plane synonyms ──
    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    # ── Anatomical / organ synonyms (SLAKE covers head/chest/abdomen/pelvis) ──
    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    # ── Abnormality synonyms ──
    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    # ── Remove articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans




# =============================================================================
# LOAD DATASET INTO RAM
# =============================================================================
print("\n" + "="*60 + "\nLOADING VQA-RAD INTO RAM\n" + "="*60)
from datasets import load_dataset
#ds = load_dataset('flaviagiammarino/vqa-rad')
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')
#ds = load_dataset('flaviagiammarino/path-vqa')

def extract_to_ram(split_data, name):
    samples = []
    for s in tqdm(split_data, desc=name):
        try:
            img = s.get('image')
            q = str(s.get('question', ''))
            a = str(s.get('answer', '')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a:
                samples.append({'image': img.convert('RGB'), 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}")
    return samples

train_samples = extract_to_ram(ds['train'], 'train')
test_samples = extract_to_ram(ds['test'], 'test')
del ds  # free HF cache

# Build vocab
all_ans = [s['answer'] for s in train_samples + test_samples]
counts = Counter(all_ans)
filtered = [(a,c) for a,c in counts.most_common() if c >= cfg.min_answer_freq]
if cfg.max_answer_vocab > 0: filtered = filtered[:cfg.max_answer_vocab]
answer_vocab = {a: i for i, (a,_) in enumerate(sorted(filtered, key=lambda x: x[0]))}
if '<unk>' not in answer_vocab: answer_vocab['<unk>'] = len(answer_vocab)
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes} classes")

# Train/val split
indices = list(range(len(train_samples))); random.shuffle(indices)
n_val = int(len(indices) * cfg.val_split)
trn = [train_samples[i] for i in indices[n_val:]]
val = [train_samples[i] for i in indices[:n_val]]
print(f"  Train: {len(trn)}, Val: {len(val)}, Test: {len(test_samples)}")

# =============================================================================
# LOAD BiomedCLIP
# =============================================================================
print("\n" + "="*60 + "\nLOADING BiomedCLIP\n" + "="*60)
from open_clip import create_model_and_transforms, get_tokenizer
clip_model, preprocess_train, preprocess_val = create_model_and_transforms(cfg.clip_model_name)
tokenizer = get_tokenizer(cfg.clip_model_name)
clip_model = clip_model.to(device)
print("  BiomedCLIP loaded")

# =============================================================================
# DATASET + DATALOADER
# =============================================================================
class VQADataset(Dataset):
    def __init__(self, samples, vocab, preprocess, tokenizer):
        self.samples = samples; self.vocab = vocab
        self.preprocess = preprocess; self.tokenizer = tokenizer
        self.unk = vocab.get('<unk>', 0)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return (self.preprocess(s['image']), self.tokenizer([s['question']])[0],
                self.vocab.get(s['answer'], self.unk))

def collate_fn(batch):
    imgs, txts, lbls = zip(*batch); imgs = torch.stack(imgs)
    mx = max(t.shape[0] for t in txts)
    padded = torch.zeros(len(txts), mx, dtype=txts[0].dtype)
    for i, t in enumerate(txts): padded[i, :t.shape[0]] = t
    return imgs, padded, torch.tensor(lbls, dtype=torch.long)

train_loader = DataLoader(VQADataset(trn, answer_vocab, preprocess_train, tokenizer),
                           batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(VQADataset(val, answer_vocab, preprocess_val, tokenizer),
                         batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(VQADataset(test_samples, answer_vocab, preprocess_val, tokenizer),
                          batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# =============================================================================
# MODEL — identical across all 3 setups
# =============================================================================
class FusionTransformerLayer(nn.Module):
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.v2t_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.v2t_norm1 = nn.LayerNorm(dim)
        self.v2t_ffn = nn.Sequential(nn.Linear(dim,dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4,dim), nn.Dropout(dropout))
        self.v2t_norm2 = nn.LayerNorm(dim)
        self.t2v_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.t2v_norm1 = nn.LayerNorm(dim)
        self.t2v_ffn = nn.Sequential(nn.Linear(dim,dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4,dim), nn.Dropout(dropout))
        self.t2v_norm2 = nn.LayerNorm(dim)
    def forward(self, v, t, text_key_padding_mask=None):
        o, _ = self.v2t_attn(v, t, t, key_padding_mask=text_key_padding_mask)
        v = self.v2t_norm1(v+o); v = self.v2t_norm2(v + self.v2t_ffn(v))
        o, _ = self.t2v_attn(t, v, v)
        t = self.t2v_norm1(t+o); t = self.t2v_norm2(t + self.t2v_ffn(t))
        return v, t

class BiomedCLIPVQAModel(nn.Module):
    def __init__(self, clip_model, cfg, num_classes):
        super().__init__()
        self.clip_model = clip_model; D = cfg.hidden_dim
        self.vis_proj = nn.Linear(cfg.vision_dim, D) if cfg.vision_dim != D else nn.Identity()
        self.txt_proj = nn.Linear(cfg.text_dim, D) if cfg.text_dim != D else nn.Identity()
        self.fusion_layers = nn.ModuleList([FusionTransformerLayer(D, cfg.num_attn_heads, cfg.fusion_dropout) for _ in range(cfg.num_fusion_layers)])
        self.pool_query = nn.Parameter(torch.randn(1,1,D)*0.02)
        self.pool_attn = nn.MultiheadAttention(D, cfg.num_attn_heads, dropout=cfg.fusion_dropout, batch_first=True)
        self.pool_norm = nn.LayerNorm(D)
        self.head = nn.Sequential(nn.Linear(D,D), nn.GELU(), nn.Dropout(cfg.fusion_dropout),
                                   nn.Linear(D,D//2), nn.GELU(), nn.Dropout(cfg.fusion_dropout), nn.Linear(D//2, num_classes))
    def encode_image(self, images):
        ve = self.clip_model.visual
        x = ve.trunk.patch_embed(images); x = ve.trunk._pos_embed(x); x = ve.trunk.patch_drop(x)
        x = ve.trunk.norm_pre(x); x = ve.trunk.blocks(x); return ve.trunk.norm(x)
    def encode_text(self, input_ids):
        te = self.clip_model.text; amask = (input_ids != 0).long()
        out = te.transformer(input_ids=input_ids, attention_mask=amask)
        return out.last_hidden_state, amask
    def forward(self, images, input_ids):
        v = self.vis_proj(self.encode_image(images))
        t, amask = self.encode_text(input_ids); t = self.txt_proj(t)
        kpm = (amask == 0)
        for fl in self.fusion_layers: v, t = fl(v, t, text_key_padding_mask=kpm)
        combined = torch.cat([v, t], dim=1); B = combined.shape[0]
        pq = self.pool_query.expand(B,-1,-1)
        pooled, _ = self.pool_attn(pq, combined, combined)
        return self.head(self.pool_norm(pq + pooled).squeeze(1))

model = BiomedCLIPVQAModel(clip_model, cfg, num_classes).to(device)
n_total = sum(p.numel() for p in model.parameters())
print(f"\n  Model: {n_total:,} params ({n_total/1e6:.1f}M)")

# =============================================================================
# TRAINING
# =============================================================================
print("\n" + "="*60 + "\nTRAINING (Centralized)\n" + "="*60)

def get_param_groups(model, cfg, frozen):
    enc = list(model.clip_model.parameters()); enc_ids = set(id(p) for p in enc)
    fh = [p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
    if frozen:
        for p in enc: p.requires_grad = False
        return [{'params': fh, 'lr': cfg.learning_rate}]
    for p in enc: p.requires_grad = True
    return [{'params': [p for p in enc if p.requires_grad], 'lr': cfg.learning_rate * cfg.encoder_lr_scale},
            {'params': fh, 'lr': cfg.learning_rate}]

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
param_groups = get_param_groups(model, cfg, frozen=True)
optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

history = {'epoch':[], 'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[],
           'test_loss':[], 'test_acc':[], 'lr':[], 'epoch_time':[]}
best_val_acc, best_state = 0.0, None
patience, lr_patience = 0, 0

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    if epoch == cfg.freeze_clip_epochs + 1:
        print(f"\n  === Unfreezing encoders ===")
        param_groups = get_param_groups(model, cfg, frozen=False)
        optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    # Train
    model.train(); tr_loss, tr_correct, tr_total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:2d}/{cfg.epochs} [train]", leave=False)
    for imgs, txts, lbls in pbar:
        imgs, txts, lbls = imgs.to(device), txts.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=scaler is not None):
            logits = model(imgs, txts); loss = criterion(logits, lbls)
        if scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tr_loss += loss.item()*lbls.size(0); tr_correct += (logits.argmax(-1)==lbls).sum().item(); tr_total += lbls.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100*tr_correct/tr_total:.1f}%")
    tr_loss /= tr_total; tr_acc = 100*tr_correct/tr_total

    # Eval
    model.eval()
    def eval_loader(loader):
        loss_sum, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, txts, lbls in loader:
                imgs, txts, lbls = imgs.to(device), txts.to(device), lbls.to(device)
                logits = model(imgs, txts); loss = criterion(logits, lbls)
                loss_sum += loss.item()*lbls.size(0); correct += (logits.argmax(-1)==lbls).sum().item(); total += lbls.size(0)
        return loss_sum/total, 100*correct/total

    va_loss, va_acc = eval_loader(val_loader)
    te_loss, te_acc = eval_loader(test_loader)
    lr = optimizer.param_groups[-1]['lr']
    elapsed = time.time() - t0

    history['epoch'].append(epoch); history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss); history['val_acc'].append(va_acc)
    history['test_loss'].append(te_loss); history['test_acc'].append(te_acc)
    history['lr'].append(lr); history['epoch_time'].append(elapsed)

    marker = ""
    if va_acc > best_val_acc:
        best_val_acc = va_acc; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience = lr_patience = 0; marker = " ★"
    else:
        patience += 1; lr_patience += 1

    print(f"Epoch {epoch:2d}/{cfg.epochs} [{elapsed:.1f}s]  "
          f"Train: {tr_loss:.4f}/{tr_acc:.1f}%  Val: {va_loss:.4f}/{va_acc:.1f}%  "
          f"Test: {te_acc:.1f}%  LR={lr:.1e}{marker}")

    if lr_patience >= cfg.lr_reduce_patience:
        for pg in optimizer.param_groups: pg['lr'] = max(pg['lr']*cfg.lr_reduce_factor, cfg.min_lr)
        lr_patience = 0; print(f"  ↓ LR → {optimizer.param_groups[-1]['lr']:.1e}")

    if patience >= cfg.early_stopping_patience:
        print(f"  Early stopping at epoch {epoch}"); break

# Final eval
if best_state: model.load_state_dict(best_state)
te_loss, te_acc = eval_loader(test_loader)
print(f"\n{'='*60}\nFINAL TEST: {te_acc:.2f}% (best val: {best_val_acc:.2f}%)\n{'='*60}")

# =============================================================================
# SAVE TO EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Centralized VQA-RAD"

# Header style
header_font = Font(name='Arial', bold=True, size=11, color='FFFFFF')
header_fill = PatternFill(start_color='1A5276', end_color='1A5276', fill_type='solid')
border = Border(bottom=Side(style='thin', color='CCCCCC'))

headers = ['Epoch', 'Train Loss', 'Train Acc (%)', 'Val Loss', 'Val Acc (%)',
           'Test Loss', 'Test Acc (%)', 'LR', 'Time (s)']
for col, h in enumerate(headers, 1):
    c = ws.cell(row=1, column=col, value=h)
    c.font = header_font; c.fill = header_fill; c.alignment = Alignment(horizontal='center')

for i, ep in enumerate(history['epoch']):
    row = i + 2
    ws.cell(row=row, column=1, value=ep)
    ws.cell(row=row, column=2, value=round(history['train_loss'][i], 4))
    ws.cell(row=row, column=3, value=round(history['train_acc'][i], 2))
    ws.cell(row=row, column=4, value=round(history['val_loss'][i], 4))
    ws.cell(row=row, column=5, value=round(history['val_acc'][i], 2))
    ws.cell(row=row, column=6, value=round(history['test_loss'][i], 4))
    ws.cell(row=row, column=7, value=round(history['test_acc'][i], 2))
    ws.cell(row=row, column=8, value=f"{history['lr'][i]:.1e}")
    ws.cell(row=row, column=9, value=round(history['epoch_time'][i], 1))

# Summary sheet
ws2 = wb.create_sheet("Summary")
summary = [
    ("Method", "Centralized"),
    ("Dataset", "VQA-RAD"),
    ("Model", "BiomedCLIP + 4-layer Fusion"),
    ("Total Params", f"{n_total:,}"),
    ("Num Classes", num_classes),
    ("Best Val Acc (%)", round(best_val_acc, 2)),
    ("Final Test Acc (%)", round(te_acc, 2)),
    ("Epochs Trained", len(history['epoch'])),
    ("Batch Size", cfg.batch_size),
    ("LR", cfg.learning_rate),
    ("Label Smoothing", cfg.label_smoothing),
    ("Freeze Epochs", cfg.freeze_clip_epochs),
]
for i, (k, v) in enumerate(summary, 1):
    ws2.cell(row=i, column=1, value=k).font = Font(bold=True, name='Arial')
    ws2.cell(row=i, column=2, value=v)

# Auto-width
for ws_sheet in [ws, ws2]:
    for col in ws_sheet.columns:
        mx = max(len(str(c.value or '')) for c in col) + 2
        ws_sheet.column_dimensions[col[0].column_letter].width = mx

xlsx_path = f"{OUTPUT_DIR}/slake_centralized_results.xlsx"
wb.save(xlsx_path)
print(f"\nResults saved to {xlsx_path}")
print("DONE!")